# LRTIA - Context Ablation Approach

**Key insight:** Long-range context provides a subtle, distributed benefit across ALL tokens, not dramatic effects on specific tokens.

**New approach:** Progressive context truncation
1. Pick a target region (last N tokens of document)
2. Measure perplexity with FULL context
3. Measure perplexity with truncated context (only last 256, 128, 64 tokens)
4. Plot how perplexity degrades as context is removed

**Prediction:**
- **Intact:** Perplexity increases as distant context is removed (it was helping!)
- **Shuffled:** Perplexity stays flat (distant context wasn't useful anyway)

The RATE of degradation IS the memory curve.

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import random
import re
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
model.eval()
print(f"Loaded {MODEL_NAME}")

In [ ]:
# Extended passages (~500-600 tokens each) - text that builds on itself
PASSAGES = [
    # Neural Networks - extended technical explanation
    """Let me explain how neural networks learn. First, we initialize the weights randomly. These weights are the parameters that the network will adjust during training. The network takes an input and multiplies it by these weights to produce an output. We then compare this output to the correct answer using a loss function. The loss tells us how wrong the network was. Using calculus, we compute the gradient of this loss with respect to each weight. The gradient points in the direction that would increase the loss, so we move the weights in the opposite direction. This process is called gradient descent. We repeat this process many times with different training examples. Gradually, the weights converge to values that minimize the loss. The network has now learned to map inputs to outputs. This is the essence of backpropagation, the algorithm that makes deep learning possible. But there is more to understand about how these networks function. The architecture of a neural network consists of layers. The first layer receives the raw input data. Each subsequent layer transforms the data further. The final layer produces the output prediction. Between the input and output are hidden layers. These hidden layers learn increasingly abstract representations of the data. Early layers might detect simple patterns like edges in an image. Deeper layers combine these simple patterns into complex features like faces or objects. This hierarchical learning is what gives deep networks their power. The number of layers and neurons per layer are hyperparameters that we must choose carefully. Too few neurons and the network cannot learn complex patterns. Too many neurons and the network might memorize the training data instead of learning general patterns. This problem is called overfitting. To prevent overfitting, we use techniques like dropout and regularization. Dropout randomly disables neurons during training, forcing the network to learn redundant representations. Regularization adds a penalty term to the loss function that discourages large weights. Together, these techniques help the network generalize to new data it has never seen before.""",

    # Detective Story - extended narrative
    """Detective Chen arrived at the crime scene just after midnight. The victim was a wealthy businessman named Richard Hayes, found slumped over his mahogany desk in the study. There were no signs of forced entry, which meant the killer was someone the victim knew and trusted enough to let inside. Chen noticed a half-empty glass of wine on the desk, the red liquid catching the lamplight. She bagged it as evidence, suspecting poison rather than violence. The security footage showed three visitors that evening: the victim's wife Eleanor, his business partner Marcus Webb, and his lawyer Thomas Grant. Chen interviewed each of them the next morning at the station. The wife claimed she left at nine after a brief argument about money. She said Richard had been stressed about a business deal gone wrong. The partner Marcus said he arrived at ten to discuss quarterly projections. He claimed Richard was alive and well when he left at eleven. The lawyer Thomas insisted he was never there at all. He said he had been home all evening with his wife. But the footage told a different story. It clearly showed Thomas entering at eleven thirty and leaving at midnight. The lawyer had lied. Chen brought him back for questioning. Under pressure, Thomas began to sweat. He admitted to being there but claimed he left Richard alive. Chen pressed harder, pointing out that Richard was dead by the time the housekeeper found him at six in the morning. The toxicology report came back showing traces of a rare poison in the wine glass. Chen discovered that Thomas had recently purchased the same compound through a shell company. Confronted with this evidence, Thomas finally broke down and confessed to the murder. Richard had discovered that Thomas was embezzling millions from the company. Thomas had poisoned the wine to protect his secret and his freedom.""",

    # Climate Change - extended scientific argument
    """The evidence for climate change is overwhelming and comes from multiple independent lines of research. Global temperatures have risen by 1.1 degrees Celsius since pre-industrial times, with most of that warming occurring in the past fifty years. This warming correlates precisely with the increase in atmospheric carbon dioxide from burning fossil fuels. Ice cores drilled from Antarctica and Greenland show that current CO2 levels are higher than at any point in 800,000 years. The physics behind this warming is well understood and has been known for over a century. Carbon dioxide and other greenhouse gases trap infrared radiation that would otherwise escape to space, warming the planet like a blanket. Climate models based on this physics predicted the current warming decades ago, and their predictions have proven remarkably accurate. The effects of this warming are already visible around the world. Sea levels are rising as ice sheets in Greenland and Antarctica melt and as warming oceans expand. Coastal cities are experiencing more frequent flooding during high tides and storms. Extreme weather events are becoming more frequent and more intense. Heat waves that once occurred once per decade now happen every few years. Hurricanes are growing stronger as they draw energy from warmer ocean waters. Species are shifting their ranges toward the poles and to higher elevations as their traditional habitats become too warm. Coral reefs are bleaching and dying as ocean temperatures rise and waters become more acidic. Every major scientific organization in the world agrees on these basic facts. The debate among scientists is not about whether climate change is happening or whether humans are causing it. That debate was settled decades ago. The remaining questions concern the precise timing and magnitude of future changes, and what we should do about them. We must act now to reduce emissions before the damage becomes irreversible and billions of people are displaced by rising seas and failing harvests.""",

    # Risotto Recipe - extended cooking instructions
    """Today I will teach you how to make the perfect risotto, a classic Italian dish that requires patience and attention but rewards you with incredible flavor and texture. Start by heating your chicken stock in a separate pot and keep it warm throughout the cooking process. This is essential because adding cold stock would shock the rice and interrupt the cooking. In your main pan, a wide shallow skillet works best, melt two tablespoons of butter over medium heat. Add one finely diced onion and cook slowly until the pieces become translucent, about five minutes. Do not let them brown, as this would add a bitter flavor to the finished dish. Now add one and a half cups of arborio rice. This special short-grain rice is essential for risotto because it contains more starch than regular rice. Toast the rice in the butter for two minutes, stirring constantly with a wooden spoon. The grains should become slightly translucent at the edges while remaining opaque in the center. Now add a generous splash of dry white wine, about half a cup. Stir until the wine is completely absorbed. This is where patience becomes essential. You will spend the next eighteen to twenty minutes stirring and adding stock. Add the warm stock one ladle at a time, stirring frequently. Wait until each addition is almost fully absorbed before adding more. You should see the rice release its starch, creating a creamy consistency. This gradual process is what makes risotto different from regular rice dishes. The constant stirring releases the starch from the outside of each grain, creating the characteristic creamy texture without any cream. After about eighteen minutes, taste the rice. It should be al dente, tender but with a slight firmness in the center. Remove the pan from heat immediately. Now stir in two more tablespoons of cold butter and a generous handful of freshly grated Parmigiano-Reggiano cheese. This final step, called mantecatura, gives the risotto its glossy finish and rich flavor. Let it rest for two minutes before serving in warm bowls.""",

    # Roman Empire - extended historical narrative
    """The fall of the Roman Empire was not a single dramatic event but a gradual process spanning centuries. Historians still debate when exactly the empire fell, but most agree that multiple factors contributed to Rome's slow decline and eventual collapse. First, the empire had simply grown too large to govern effectively with ancient technology. At its height, Rome controlled territories from Britain to Mesopotamia, from the Rhine to the Sahara. Communication across such vast distances was painfully slow. Messages could take weeks or months to travel from frontier provinces to Rome. This made it nearly impossible to respond quickly to invasions or rebellions. Emperors often learned of crises only after they had spiraled out of control. Second, the Roman army changed fundamentally over the centuries. Early legions were composed of citizen soldiers who owned property and had personal stakes in Rome's success. By the late empire, the army increasingly relied on Germanic mercenaries who had little loyalty to Rome itself. These soldiers fought for pay and plunder, not for patriotic ideals. Sometimes they turned against the emperors who hired them. Third, economic troubles plagued the later empire. Constant warfare drained the treasury. Emperors debased the currency by reducing the silver content of coins, leading to rampant inflation. Trade declined as roads fell into disrepair and pirates infested the seas. The western provinces became impoverished while the eastern empire remained relatively wealthy. Constantine's decision to move the capital to Constantinople in 330 AD accelerated this divide between east and west. When Germanic tribes finally sacked Rome itself in 410 AD, the event shocked the Mediterranean world. The eternal city had not been conquered in eight hundred years. Yet Rome had already lost much of its former glory by then. The last western Roman emperor, a teenager named Romulus Augustulus, was deposed in 476 AD. This date is often cited as the fall of Rome, though it is somewhat arbitrary. The eastern empire, which historians call the Byzantine Empire, would survive and often thrive for another thousand years until Constantinople fell to the Ottoman Turks in 1453.""",
]

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]

print(f"Loaded {len(PASSAGES)} passages")
for i, p in enumerate(PASSAGES):
    n_tokens = len(tokenizer.encode(p))
    n_sentences = len(split_sentences(p))
    print(f"  Passage {i+1}: {n_tokens} tokens, {n_sentences} sentences")

In [ ]:
@torch.no_grad()
def compute_perplexity_on_region(token_ids, target_start, target_end):
    """
    Compute perplexity only on tokens in [target_start, target_end].
    Context before target_start is provided but not scored.
    """
    if target_end > len(token_ids):
        target_end = len(token_ids)
    
    input_ids = torch.tensor([token_ids], device=model.device)
    
    # Get all logits
    outputs = model(input_ids)
    logits = outputs.logits[0]  # [seq_len, vocab]
    
    # Compute loss only on target region
    # For position i, we predict token i+1
    total_loss = 0.0
    count = 0
    
    for i in range(target_start, target_end - 1):
        # Predict token at position i+1 given context up to position i
        log_probs = torch.log_softmax(logits[i], dim=-1)
        target_token = token_ids[i + 1]
        token_loss = -log_probs[target_token].item()
        total_loss += token_loss
        count += 1
    
    if count == 0:
        return float('inf'), 0
    
    avg_loss = total_loss / count
    perplexity = np.exp(avg_loss)
    
    return perplexity, avg_loss

In [ ]:
def analyze_context_dependency(text, context_lengths, target_size=50):
    """
    Measure perplexity on the last `target_size` tokens
    with varying amounts of preceding context.
    
    context_lengths: list of how many tokens of context to provide
                     (in addition to the target region)
    """
    full_tokens = tokenizer.encode(text)
    n_tokens = len(full_tokens)
    
    if n_tokens < target_size + max(context_lengths):
        print(f"Warning: text too short ({n_tokens} tokens)")
        return None
    
    # Target region is always the last `target_size` tokens
    target_start_in_full = n_tokens - target_size
    target_end_in_full = n_tokens
    
    results = []
    
    for ctx_len in context_lengths:
        # How much of the document to include
        # We want `ctx_len` tokens BEFORE the target region
        doc_start = max(0, target_start_in_full - ctx_len)
        
        # Extract this portion
        truncated_tokens = full_tokens[doc_start:]
        
        # Target region position within truncated sequence
        target_start = len(truncated_tokens) - target_size
        target_end = len(truncated_tokens)
        
        ppl, loss = compute_perplexity_on_region(truncated_tokens, target_start, target_end)
        
        actual_context = target_start  # Actual context provided
        
        results.append({
            'requested_context': ctx_len,
            'actual_context': actual_context,
            'perplexity': ppl,
            'loss': loss,
        })
    
    return results

In [ ]:
# Context lengths to test (tokens before target region)
# Extended passages are ~500 tokens, so max usable context is ~460
CONTEXT_LENGTHS = [8, 16, 32, 48, 64, 96, 128, 192, 256, 320, 384]
TARGET_SIZE = 40  # Last 40 tokens are the target region

all_results = []

for i, passage in enumerate(tqdm(PASSAGES, desc="Processing passages")):
    passage_name = ['NN', 'Detective', 'Climate', 'Recipe', 'Rome'][i]
    
    # INTACT
    intact_text = " ".join(passage.split())
    intact_results = analyze_context_dependency(intact_text, CONTEXT_LENGTHS, TARGET_SIZE)
    
    if intact_results:
        for r in intact_results:
            r['passage'] = passage_name
            r['condition'] = 'intact'
            all_results.append(r)
    
    # SHUFFLED (multiple versions)
    sentences = split_sentences(passage)
    for seed in range(3):
        rng = random.Random(42 + i * 100 + seed)
        shuffled = sentences.copy()
        rng.shuffle(shuffled)
        shuffled_text = " ".join(shuffled)
        
        shuffled_results = analyze_context_dependency(shuffled_text, CONTEXT_LENGTHS, TARGET_SIZE)
        
        if shuffled_results:
            for r in shuffled_results:
                r['passage'] = passage_name
                r['condition'] = 'shuffled'
                r['seed'] = seed
                all_results.append(r)

df = pd.DataFrame(all_results)
print(f"\nCollected {len(df)} measurements")
if len(df) > 0:
    print(f"Context lengths tested: {sorted(df['actual_context'].unique())}")
else:
    print("ERROR: No results collected. Check passage lengths.")

In [ ]:
print("=" * 70)
print("RESULTS: Perplexity vs Context Length")
print("=" * 70)

# Average across passages
pivot = df.pivot_table(
    values='perplexity', 
    index='actual_context', 
    columns='condition', 
    aggfunc='mean'
)
print("\nPerplexity by context length:")
print(pivot.round(2))

In [ ]:
# THE KEY VISUALIZATION
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Perplexity vs Context Length
ax = axes[0]
colors = {'intact': '#2ecc71', 'shuffled': '#e74c3c'}

for condition in ['intact', 'shuffled']:
    cond_df = df[df['condition'] == condition]
    means = cond_df.groupby('actual_context')['perplexity'].mean()
    stds = cond_df.groupby('actual_context')['perplexity'].std()
    sems = stds / np.sqrt(cond_df.groupby('actual_context').size())
    
    ax.errorbar(means.index, means.values, yerr=sems.values,
                marker='o', capsize=4, label=condition, 
                color=colors[condition], linewidth=2, markersize=8)

ax.set_xlabel('Context Length (tokens before target)', fontsize=12)
ax.set_ylabel('Perplexity on Target Region', fontsize=12)
ax.set_title('How Much Does Context Help?', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Right: The "Memory Curve" - Perplexity REDUCTION from adding context
ax = axes[1]

for condition in ['intact', 'shuffled']:
    cond_df = df[df['condition'] == condition]
    means = cond_df.groupby('actual_context')['perplexity'].mean().sort_index()
    
    # Baseline: perplexity with minimal context
    baseline = means.iloc[0]
    
    # Reduction from baseline
    reduction = baseline - means
    
    ax.plot(reduction.index, reduction.values,
            marker='o', label=condition, 
            color=colors[condition], linewidth=2, markersize=8)

ax.set_xlabel('Context Length (tokens)', fontsize=12)
ax.set_ylabel('Perplexity Reduction (from minimal context)', fontsize=12)
ax.set_title('"Memory Curve": Benefit of Adding Context', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('context_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nInterpretation:")
print("- Left plot: Lower = better prediction")
print("- Right plot: Higher = more benefit from context")
print("- If intact curve is ABOVE shuffled on right plot,")
print("  intact text benefits MORE from long-range context.")

In [ ]:
# HALF-LIFE COMPUTATION
# The key scalar metric: context length to achieve 50% of total benefit

def compute_half_life(contexts, perplexities, percentile=0.5):
    """
    Compute the context length at which `percentile` of total benefit is achieved.
    Returns interpolated value for precision.
    """
    contexts = np.array(contexts)
    perplexities = np.array(perplexities)
    
    # Sort by context
    order = np.argsort(contexts)
    contexts = contexts[order]
    perplexities = perplexities[order]
    
    min_ppl = perplexities[0]   # Perplexity with minimal context
    max_ppl = perplexities[-1]  # Perplexity with maximal context
    total_benefit = min_ppl - max_ppl
    
    if total_benefit <= 0:
        return np.nan  # No improvement
    
    target_ppl = min_ppl - percentile * total_benefit
    
    # Find where we cross the target
    for i in range(len(perplexities) - 1):
        if perplexities[i] >= target_ppl >= perplexities[i+1]:
            # Linear interpolation
            frac = (perplexities[i] - target_ppl) / (perplexities[i] - perplexities[i+1])
            return contexts[i] + frac * (contexts[i+1] - contexts[i])
    
    return contexts[-1]  # Never reached target

print("=" * 70)
print("HALF-LIFE ANALYSIS (context tokens to reach 50% of benefit)")
print("=" * 70)

half_lives = {}

for condition in ['intact', 'shuffled']:
    cond_df = df[df['condition'] == condition]
    means = cond_df.groupby('actual_context')['perplexity'].mean().sort_index()
    
    contexts = means.index.values
    perplexities = means.values
    
    hl_50 = compute_half_life(contexts, perplexities, 0.5)
    hl_25 = compute_half_life(contexts, perplexities, 0.25)
    hl_75 = compute_half_life(contexts, perplexities, 0.75)
    hl_90 = compute_half_life(contexts, perplexities, 0.90)
    
    half_lives[condition] = {
        '25%': hl_25, '50%': hl_50, '75%': hl_75, '90%': hl_90
    }
    
    total_benefit = perplexities[0] - perplexities[-1]
    
    print(f"\n{condition.upper()}:")
    print(f"  Total perplexity reduction: {total_benefit:.2f}")
    print(f"  25% of benefit at: {hl_25:.0f} tokens")
    print(f"  50% of benefit at: {hl_50:.0f} tokens  <-- HALF-LIFE")
    print(f"  75% of benefit at: {hl_75:.0f} tokens")
    print(f"  90% of benefit at: {hl_90:.0f} tokens")

print(f"\n" + "-" * 50)
print(f"HALF-LIFE COMPARISON:")
print(f"  Intact:   {half_lives['intact']['50%']:.0f} tokens")
print(f"  Shuffled: {half_lives['shuffled']['50%']:.0f} tokens")
print(f"  Ratio:    {half_lives['shuffled']['50%'] / half_lives['intact']['50%']:.2f}x")
print(f"\nInterpretation: Shuffled needs {half_lives['shuffled']['50%'] / half_lives['intact']['50%']:.1f}x more context to get the same benefit.")

In [ ]:
# Statistical comparison at each context length
from scipy import stats

print("\n" + "=" * 70)
print("STATISTICAL COMPARISON: Intact vs Shuffled at Each Context Length")
print("=" * 70)

context_lengths = sorted(df['actual_context'].unique())

for ctx in context_lengths:
    intact_vals = df[(df['condition']=='intact') & (df['actual_context']==ctx)]['perplexity']
    shuffled_vals = df[(df['condition']=='shuffled') & (df['actual_context']==ctx)]['perplexity']
    
    if len(intact_vals) >= 2 and len(shuffled_vals) >= 2:
        t, p = stats.ttest_ind(intact_vals, shuffled_vals)
        diff = shuffled_vals.mean() - intact_vals.mean()
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"Context {ctx:3d}: intact={intact_vals.mean():.2f}, shuffled={shuffled_vals.mean():.2f}, diff={diff:+.2f}, p={p:.4f} {sig}")

In [ ]:
# Per-passage breakdown
print("\n" + "=" * 70)
print("PER-PASSAGE BREAKDOWN")
print("=" * 70)

for passage in df['passage'].unique():
    print(f"\n{passage}:")
    pass_df = df[df['passage'] == passage]
    
    for condition in ['intact', 'shuffled']:
        cond_df = pass_df[pass_df['condition'] == condition]
        means = cond_df.groupby('actual_context')['perplexity'].mean().sort_index()
        
        min_ppl = means.iloc[0]
        max_ppl = means.iloc[-1]
        benefit = min_ppl - max_ppl
        
        print(f"  {condition}: {min_ppl:.1f} → {max_ppl:.1f} (benefit: {benefit:.1f})")

In [ ]:
# Save results
df.to_csv('context_ablation_results.csv', index=False)
print("\nSaved to context_ablation_results.csv")

try:
    from google.colab import files
    files.download('context_ablation_results.csv')
    files.download('context_ablation.png')
except:
    pass

## Interpretation

**What we expect:**

1. **Both curves should decrease** as context increases (more context → better prediction)

2. **Intact should decrease MORE steeply** (benefits more from distant context)

3. **Shuffled should plateau earlier** (distant context doesn't help much)

4. **The GAP between curves should WIDEN at longer contexts**
   - At short context: similar (local patterns are preserved in both)
   - At long context: intact pulls ahead (global coherence kicks in)

**The "memory curve" (right plot):**
- Shows cumulative benefit of adding context
- Intact should climb higher than shuffled
- The area between curves = value of long-range coherence